## Dummy 实现

In [1]:
GPT_CONFIG_124M = {
    'vocab_size': 50257, # 字典大小，BPE 分词器使用的
    'context_length': 1024, # 上下文长度，模型所能处理的最大输入 token 数
    'emb_dim': 768, # 嵌入维度，将每个 token 转为 768维度的向量
    'n_heads': 12,  # number of attention heads
    'n_layers': 12, # number of layers，transformer 模块的层数
    'drop_rate': 0.1,  # dropout rate，0.1表示丢失 10% 的隐藏单元，用于防止过拟合
    'qkv_bias': False,  # query-key-value bias，用于决定是否在多头注意力的查询、键、值的线性层中加入偏置向量
    # 最初会禁用该选项，以遵循现代大语言模型的标准，之后在第 6 章加载 OpenAI 预训练的 GPT-2 权重时再重新考虑该设置。
}

In [2]:
import torch
import torch.nn as nn

In [3]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # token嵌入
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim']) # 每个元素从标准正态分布中随机初始化
        # 位置嵌入
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        # dropout
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        # 多个 transformer 模块
        self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        # 最终的层归一化
        self.final_norm = DummyLayerNorm(cfg['emb_dim'])
        # 线性输出层
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    # forward 方法定义了数据在模型中的流动方式
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        # 计算输入索引的 token 嵌入 和 位置嵌入
        tok_embeds = self.tok_emb(in_idx) # 获取每个 idx 对应行数的向量
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device)) # device=in_idx.device 确保位置编号和输入位于同一设备，例如都在 CPU 或 GPU 上，避免设备不一致报错。
        x = tok_embeds + pos_embeds

        # 应用 dropout
        x = self.drop_emb(x)
        # 通过 transformer block 处理数据
        x = self.trf_blocks(x)
        # 应用归一化
        x = self.final_norm(x)
        # 通过线性输出层生成 logits
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding('gpt2')
batch = []
txt1 = 'Every effort moves you'
txt2 = 'Every day holds a'

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)  # 把多个样本拼成一个 Batch。；dim=0 是什么意思    新增维度放在最前面。
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [5]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch) # 把一批 token 输入模型，执行一次前向传播，并取得模型输出。
print('output shape:', logits.shape)
print(logits)  # 输出的张量有两行，每行对应一段文本，每段文本包含4个token，每个token是一个50257维的向量，维度大小与分词器的词汇表相同

output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)
